# 🎵 音符マット配置ツール — MIDI自動変換 + 精度点検

YouTubeのURL（ピアノカバー推奨）を入力すると、
**音符マット配置ツール**にそのまま貼れる `notes` 配列を出力します。

また、既存の `notes` データと原曲音声を**ピアノロールで重ねて比較**し、
どこがずれているかを自動で可視化します。

## セル構成
| セル | 内容 | 必須 |
|---|---|---|
| ①② | インストール・ヘルパー | ✅ |
| ③ | 設定入力 | ✅ |
| ④⑤⑥ | 音声取得→解析→notes出力 | 新規追加時 |
| ⑦ | MIDIを耳で確認 | 任意 |
| ⑧ | MIDIダウンロード | 任意 |
| **⑨** | **既存notesデータと原曲の精度点検** | **修正確認時** |
| **⑩** | **ずれ箇所リスト出力** | **修正確認時** |

In [ ]:
# ① 依存ライブラリのインストール（初回のみ数分かかります）
!pip install -q yt-dlp basic-pitch matplotlib
!apt-get install -q ffmpeg fluidsynth
print('✅ インストール完了')

In [ ]:
# ② ヘルパー関数
import os, json, numpy as np
import librosa, soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
NOTE_ORDER = ['C3','C#3','D3','D#3','E3','F3','F#3','G3','G#3','A3','A#3','B3',
              'C4','C#4','D4','D#4','E4','F4','F#4','G4','G#4','A4','A#4','B4',
              'C5','C#5']

def midi_num_to_name(n):
    return f"{NOTE_NAMES[n % 12]}{n // 12 - 1}"

def note_to_midi(name):
    if not name: return None
    base = name[:-1]
    oct_ = int(name[-1])
    if base not in NOTE_NAMES: return None
    return NOTE_NAMES.index(base) + (oct_ + 1) * 12

def transpose_to_game_range(note_name):
    if not note_name: return None
    base = note_name[:-1]
    oct_ = int(note_name[-1])
    while oct_ > 4 or (oct_ == 5 and base not in ('C', 'C#')):
        oct_ -= 1
    while oct_ < 3:
        oct_ += 1
    return f"{base}{oct_}"

def note_events_to_onpu(note_events, bpm):
    events = sorted(note_events, key=lambda x: x[0])
    WINDOW = 0.05
    groups = []
    i = 0
    while i < len(events):
        s0 = events[i][0]
        best = events[i]
        j = i + 1
        while j < len(events) and events[j][0] - s0 <= WINDOW:
            if events[j][2] > best[2]: best = events[j]
            j += 1
        groups.append(best)
        i = j

    beat_sec = 60.0 / bpm
    eighth_sec = beat_sec * 0.5
    notes_out = []
    prev_end = 0.0
    for (start, end, pitch_midi, amp, bends) in groups:
        note_name = transpose_to_game_range(midi_num_to_name(int(pitch_midi)))
        gap = start - prev_end
        if gap > eighth_sec * 0.6:
            rd = max(0.5, round(gap / beat_sec / 0.5) * 0.5)
            if notes_out and notes_out[-1]['p'] is None:
                notes_out[-1]['d'] = round(notes_out[-1]['d'] + rd, 1)
            else:
                notes_out.append({'p': None, 'd': rd})
        dur = end - start
        d = max(0.5, round(dur / beat_sec / 0.5) * 0.5)
        notes_out.append({'p': note_name, 'd': d})
        prev_end = end
    return notes_out

def format_notes_js(notes, per_line=8):
    lines = []
    line = []
    for n in notes:
        if n['p'] is None:
            entry = f"r({n['d']})" if n['d'] != 1 else "r()"
        elif n['d'] == 1:
            entry = f"p('{n['p']}')"
        else:
            entry = f"p('{n['p']}',{n['d']})"
        line.append(entry)
        if len(line) >= per_line:
            lines.append('      ' + ','.join(line) + ',')
            line = []
    if line:
        lines.append('      ' + ','.join(line) + ',')
    return '\n'.join(lines)

def onpu_notes_to_timeline(notes, bpm, offset_sec=0):
    """onpu notes配列 → (start_sec, end_sec, pitch_name) のリスト"""
    beat_sec = 60.0 / bpm
    timeline = []
    t = offset_sec
    for n in notes:
        dur = n['d'] * beat_sec
        if n['p'] is not None:
            timeline.append((t, t + dur, n['p']))
        t += dur
    return timeline

print('✅ ヘルパー関数 読み込み完了')

In [ ]:
# ③ ここを編集してください
# ───────────────────────────────────────────────
YOUTUBE_URL = 'https://www.youtube.com/watch?v=XXXXXXXXX'  # ← ピアノカバーのURL
SONG_ID     = 'maribako'
SONG_TITLE  = 'まり箱'
ARTIST      = '宝鐘マリン'
CATEGORY    = 'marine'
BPM         = 130
START_SEC   = 0    # イントロをスキップする場合は秒数を指定
END_SEC     = 60   # None にすると全部
# ───────────────────────────────────────────────
print(f'設定: {SONG_TITLE} / BPM={BPM} / {START_SEC}〜{END_SEC}秒')

In [ ]:
# ④ YouTube から音声をダウンロード
import yt_dlp

WAV_PATH = f'/tmp/{SONG_ID}.wav'
ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': f'/tmp/{SONG_ID}.%(ext)s',
    'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav', 'preferredquality': '0'}],
    'quiet': True, 'no_warnings': True,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info(YOUTUBE_URL, download=True)
    print(f'✅ ダウンロード完了: {info["title"]} ({info.get("duration","?")} 秒)')

In [ ]:
# ⑤ basic-pitch で音声→MIDI変換
from basic_pitch.inference import predict
from basic_pitch import ICASSP_2022_MODEL_PATH

print('🎵 音声を解析中... (1〜2分かかります)')
y, sr = librosa.load(WAV_PATH, sr=22050, mono=True,
                     offset=START_SEC,
                     duration=(END_SEC - START_SEC) if END_SEC else None)
CLIP_PATH = f'/tmp/{SONG_ID}_clip.wav'
sf.write(CLIP_PATH, y, sr)

model_output, midi_data, note_events = predict(
    CLIP_PATH, ICASSP_2022_MODEL_PATH,
    onset_threshold=0.5, frame_threshold=0.3,
    minimum_note_length=58,
    minimum_frequency=librosa.note_to_hz('C3'),
    maximum_frequency=librosa.note_to_hz('C6'),
)
print(f'✅ 検出完了: {len(note_events)} 音符イベント')

In [ ]:
# ⑥ onpu-mat-tool 形式に変換して出力
notes = note_events_to_onpu(note_events, BPM)
MIDI_OUT = f'/tmp/{SONG_ID}.mid'
midi_data.write(MIDI_OUT)

js_notes = format_notes_js(notes)
print('=' * 60)
print('📋 以下をコピーして songs_public.js か songs_local.js に追加:')
print('=' * 60)
print(f"""  {{
    id:'{SONG_ID}', title:'{SONG_TITLE}', artist:'{ARTIST}',
    category:'{CATEGORY}', bpm:{BPM},
    notes:[
{js_notes}
    ],
    markers:[]
  }},""")
print('=' * 60)
print(f'総音符数: {len(notes)}')

In [ ]:
# ⑦ 【任意】生成したMIDIを再生して耳で確認
from IPython.display import Audio
!fluidsynth -ni /usr/share/sounds/sf2/FluidR3_GM.sf2 {MIDI_OUT} -F /tmp/preview.wav -r 22050 2>/dev/null
Audio('/tmp/preview.wav')

In [ ]:
# ⑧ 【任意】MIDIファイルをダウンロード
from google.colab import files
files.download(MIDI_OUT)
print('→ ツールの「♪+」ボタンにドロップすれば直接使えます')

---
## 🔍 精度点検セクション
既存の `notes` データが原曲と合っているかをピアノロールで視覚確認します。

- **青いブロック** = 原曲音声（basic-pitchで解析）
- **赤いブロック** = ツールの現在のnotesデータ
- **重なっている部分** = 一致している
- **ずれている部分** = 修正候補

In [ ]:
# ⑨ 点検設定 — ここを編集してください
# ───────────────────────────────────────────────
# 点検する音声（セル④でダウンロード済みなら同じURLでOK）
CHECK_WAV = WAV_PATH  # または '/tmp/別ファイル.wav'

# ツールに現在登録されているnotesデータを貼り付ける
# index.html の notes:[...] の中身をそのままコピー
CURRENT_NOTES_JS = """
p('E4'),p('E4'),p('F4'),p('G4'),
p('G4'),p('F4'),p('E4'),p('D4'),
p('C4'),p('C4'),p('D4'),p('E4'),
p('E4',1.5),p('D4',0.5),p('D4',2),
"""
# 点検区間（秒）
CHECK_START = START_SEC
CHECK_END   = END_SEC
# ───────────────────────────────────────────────
print('設定OK')

In [ ]:
# ⑩ 精度点検 — ピアノロール比較図を生成
import re

# --- A. 現在のnotesデータをパース ---
def parse_notes_js(js_text):
    """p('X',d) / r(d) 形式をパース"""
    notes = []
    for m in re.finditer(r"([pr])\('?([^'\)]*)'?(?:,([\d.]+))?\)", js_text):
        kind, pitch, d = m.group(1), m.group(2), m.group(3)
        d = float(d) if d else 1.0
        if kind == 'r':
            notes.append({'p': None, 'd': d})
        else:
            notes.append({'p': pitch, 'd': d})
    return notes

current_notes = parse_notes_js(CURRENT_NOTES_JS)
tool_timeline = onpu_notes_to_timeline(current_notes, BPM, offset_sec=0)
print(f'ツールのnotes: {len(current_notes)} 音符')

# --- B. 音声を解析（basic-pitch）---
print('🎵 原曲を解析中...')
y_check, sr_check = librosa.load(
    CHECK_WAV, sr=22050, mono=True,
    offset=CHECK_START,
    duration=(CHECK_END - CHECK_START) if CHECK_END else None
)
CHECK_CLIP = '/tmp/check_clip.wav'
sf.write(CHECK_CLIP, y_check, sr_check)

_, _, ref_events = predict(
    CHECK_CLIP, ICASSP_2022_MODEL_PATH,
    onset_threshold=0.4, frame_threshold=0.25,
    minimum_note_length=50,
    minimum_frequency=librosa.note_to_hz('C3'),
    maximum_frequency=librosa.note_to_hz('C6'),
)

# 最高音のみ抽出（メロディーライン）
WINDOW = 0.05
ref_sorted = sorted(ref_events, key=lambda x: x[0])
ref_melody = []
i = 0
while i < len(ref_sorted):
    s0 = ref_sorted[i][0]
    best = ref_sorted[i]
    j = i + 1
    while j < len(ref_sorted) and ref_sorted[j][0] - s0 <= WINDOW:
        if ref_sorted[j][2] > best[2]: best = ref_sorted[j]
        j += 1
    name = transpose_to_game_range(midi_num_to_name(int(best[2])))
    ref_melody.append((best[0], best[1], name))
    i = j
print(f'原曲解析: {len(ref_melody)} 音符')

# --- C. ピアノロール描画 ---
NOTE_Y = {n: i for i, n in enumerate(NOTE_ORDER)}

fig, axes = plt.subplots(2, 1, figsize=(18, 8), sharex=True)
fig.suptitle(f'{SONG_TITLE} — 精度点検', fontsize=14, fontweight='bold')

# 上段: 原曲（basic-pitch解析）
ax1 = axes[0]
ax1.set_title('原曲（音声解析）', fontsize=11)
for (s, e, name) in ref_melody:
    y_pos = NOTE_Y.get(name, -1)
    if y_pos >= 0:
        ax1.barh(y_pos, e - s, left=s, height=0.7, color='#3498db', alpha=0.8)
ax1.set_yticks(range(len(NOTE_ORDER)))
ax1.set_yticklabels(NOTE_ORDER, fontsize=7)
ax1.grid(axis='x', alpha=0.3)
ax1.set_ylabel('音程')

# 下段: ツールのnotesデータ
ax2 = axes[1]
ax2.set_title('ツールのnotesデータ（現在の登録内容）', fontsize=11)
for (s, e, name) in tool_timeline:
    y_pos = NOTE_Y.get(name, -1)
    if y_pos >= 0:
        ax2.barh(y_pos, e - s, left=s, height=0.7, color='#e74c3c', alpha=0.8)
ax2.set_yticks(range(len(NOTE_ORDER)))
ax2.set_yticklabels(NOTE_ORDER, fontsize=7)
ax2.grid(axis='x', alpha=0.3)
ax2.set_xlabel('時間（秒）')
ax2.set_ylabel('音程')

plt.tight_layout()
plt.savefig('/tmp/piano_roll_compare.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ ピアノロール出力完了')

In [ ]:
# ⑪ ずれ箇所を数値でリストアップ
beat_sec = 60.0 / BPM

print(f'{'拍':>6}  {'原曲':>8}  {'ツール':>8}  {'判定':>6}')
print('-' * 40)

# 拍単位で比較（0.5拍ずつ）
max_beats = max(
    ref_melody[-1][1] / beat_sec if ref_melody else 0,
    tool_timeline[-1][1] / beat_sec if tool_timeline else 0
)

def note_at(timeline, t):
    for (s, e, name) in timeline:
        if s <= t < e: return name
    return None

mismatch = []
beat = 0.0
while beat <= max_beats:
    t = beat * beat_sec
    ref_n  = note_at(ref_melody, t)
    tool_n = note_at(tool_timeline, t)
    match = '✅' if ref_n == tool_n else ('⚠️ ズレ' if ref_n and tool_n else '–')
    if ref_n != tool_n and (ref_n or tool_n):
        mismatch.append((beat, ref_n, tool_n))
        print(f'{beat:>6.1f}  {str(ref_n):>8}  {str(tool_n):>8}  {match}')
    beat += 0.5

print()
print(f'ずれ箇所: {len(mismatch)} 箇所 / 全{int(max_beats/0.5)} 拍')
match_rate = 100 * (1 - len(mismatch) / max(1, int(max_beats/0.5)))
print(f'一致率: {match_rate:.1f}%')

In [ ]:
# ⑫ 【任意】点検図をダウンロード
from google.colab import files
files.download('/tmp/piano_roll_compare.png')
print('✅ piano_roll_compare.png をダウンロードしました')